# Plant Identification Using EfficientNetV2S with Background Removal and Explicit Botanical Feature Fusion

### Complete Google Colab (Tesla T4 GPU) Research Pipeline
This notebook implements an end-to-end botanical plant species identification framework:
1. **Environment Setup & Dependency Check**: Automated CUDA/cuDNN configuration for Tesla T4 GPU.
2. **Dataset Acquisition & Preprocessing**: Downloads and cleans the LeafSnap dataset (~30k images across 185 tree species), removing binary mask duplicates.
3. **GPU-Accelerated Background Removal**: Robust U-2-Net (`rembg`) background segmentation using ONNX Runtime with custom cuDNN settings to eliminate heuristic query crashes.
4. **Explicit Botanical Feature Extraction**: Extracts 44 domain-specific botanical leaf features (14 Shape, 13 Texture, 12 Color, and 5 Vein structural features).
5. **Data Engineering & Feature Scaling**: Stratified 70/15/15 dataset splitting with `StandardScaler`.
6. **Deep Learning Architectures**: EfficientNetV2S Baseline vs. EfficientNetV2S + Botanical Feature Fusion.
7. **Optimized Training on Tesla T4**: Mixed Precision (`mixed_float16`), XLA compilation, and two-stage transfer learning.
8. **Comprehensive Evaluation & Visual Prediction**: Accuracy, Macro/Weighted F1, Top-5 accuracy, confusion matrices, and single-leaf inference with visual explanation.

In [ ]:
# ============================================================
# 1. COLAB T4 ENVIRONMENT SETUP & PACKAGES
# ============================================================

import os
import sys
import subprocess
import gc
import json
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
import multiprocessing

print("=" * 70)
print("INSTALLING / VERIFYING DEPENDENCIES (TESLA T4 COMPATIBLE)")
print("=" * 70)

# Remove conflicting CPU onnxruntime packages if present in Colab
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "onnxruntime"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

packages = [
    "kagglehub",
    "rembg[gpu]",
    "onnxruntime-gpu",
    "scikit-learn",
    "scikit-image==0.25.2",
    "seaborn",
    "joblib",
    "pillow"
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    check=True
)

# Configure NVIDIA dynamic library paths for ONNX Runtime CUDA provider
import site
import glob

site_packages = site.getsitepackages()[0]
nvidia_lib_dirs = glob.glob(os.path.join(site_packages, "nvidia", "*", "lib"))
existing_ld = os.environ.get("LD_LIBRARY_PATH", "")
new_paths = [d for d in nvidia_lib_dirs if os.path.isdir(d)]
if new_paths:
    os.environ["LD_LIBRARY_PATH"] = ":".join(new_paths) + ":" + existing_ld

print("✓ Packages installed & environment configured")

# ============================================================
# 2. HARDWARE & ACCELERATOR CHECK
# ============================================================

import tensorflow as tf
from tensorflow.keras import mixed_precision

print("\n" + "=" * 70)
print("HARDWARE CHECK")
print("=" * 70)
print(f"CPU cores: {multiprocessing.cpu_count()}")

gpus = tf.config.list_physical_devices("GPU")
print(f"TensorFlow GPUs: {gpus}")

if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass
    print("✓ Tesla T4 GPU detected & memory growth enabled")
else:
    print("❌ WARNING: NO GPU DETECTED. In Colab, go to Runtime > Change runtime type > T4 GPU!")

try:
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(smi.stdout)
except Exception as e:
    print("nvidia-smi notice:", e)

# Enable mixed precision for Tesla T4 (FP16)
mixed_precision.set_global_policy("mixed_float16")
print(f"Mixed precision policy: {mixed_precision.global_policy()}")

# ============================================================
# 3. GLOBAL CONFIGURATION & DIRECTORIES
# ============================================================

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

IMG_SIZE = 224
BATCH_SIZE = 32  # Optimized for Tesla T4 GPU
EPOCHS = 20
FINE_TUNE_EPOCHS = 10
INITIAL_LR = 1e-3
FINE_TUNE_LR = 1e-5
DROPOUT = 0.3
AUTOTUNE = tf.data.AUTOTUNE

BASE_DIR = Path("/content/leafsnap_project")
RAW_DIR = BASE_DIR / "raw"
PROCESSED_DIR = BASE_DIR / "processed"
FEATURES_DIR = BASE_DIR / "features"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

for directory in [BASE_DIR, RAW_DIR, PROCESSED_DIR, FEATURES_DIR, MODELS_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {BASE_DIR}")

print("Setup complete. Next run the separate dataset download cell.")


INSTALLING / VERIFYING DEPENDENCIES (TESLA T4 COMPATIBLE)
✓ Packages installed & environment configured

HARDWARE CHECK
CPU cores: 2
TensorFlow GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✓ Tesla T4 GPU detected & memory growth enabled
Wed Oct 28 18:29:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:0

## Dataset download (run after setup)
This is separated from package installation so you can see whether time is spent installing dependencies or downloading LeafSnap. The first download may take several minutes; subsequent runs can use KaggleHub's cache.


In [ ]:
# ============================================================
# 2. DOWNLOAD LEAFSNAP DATASET & BUILD IMAGE INDEX
# ============================================================
# Public Kaggle dataset. First download can take several minutes.
# KaggleHub stores the download in its cache so later runs can reuse it.

import os
import time
from pathlib import Path
import pandas as pd

print("Starting LeafSnap dataset download/lookup...", flush=True)
print("If this is the first run, KaggleHub must download and unpack the dataset.", flush=True)

import kagglehub
t0 = time.time()
dataset_root = kagglehub.dataset_download("xhlulu/leafsnap-dataset")
print(f"Dataset available at: {dataset_root}", flush=True)
print(f"Download/cache lookup took {(time.time()-t0)/60:.1f} minutes", flush=True)

path_obj = Path(dataset_root)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Build one filename index once; avoids repeatedly searching the whole dataset.
print("Indexing image files (one pass)...", flush=True)
all_files = [p for p in path_obj.rglob("*") if p.is_file()]
image_files = [
    p for p in all_files
    if p.suffix.lower() in IMAGE_EXTENSIONS
    and "segmented" not in {part.lower() for part in p.parts}
]
by_name = {}
for p in image_files:
    by_name.setdefault(p.name.lower(), []).append(p)

meta_files = list(path_obj.glob("**/leafsnap-dataset-images.txt")) + list(path_obj.glob("**/leafsnap-dataset-images.csv"))
raw_df = None

if meta_files:
    meta_file = meta_files[0]
    print(f"Reading metadata: {meta_file}", flush=True)
    sep = "\t" if meta_file.suffix.lower() == ".txt" else ","
    meta_df = pd.read_csv(meta_file, sep=sep)
    valid_rows = []
    for _, row in meta_df.iterrows():
        img_rel = row.get("image_path")
        species = row.get("species")
        if pd.isna(img_rel) or pd.isna(species):
            continue
        full_p = path_obj / str(img_rel)
        if full_p.is_file() and "segmented" not in {part.lower() for part in full_p.parts}:
            valid_rows.append({"image_path": str(full_p), "label": str(species)})
        else:
            matches = by_name.get(Path(str(img_rel)).name.lower(), [])
            if matches:
                valid_rows.append({"image_path": str(matches[0]), "label": str(species)})
    if valid_rows:
        raw_df = pd.DataFrame(valid_rows)

if raw_df is None or raw_df.empty:
    print("Metadata index unavailable; deriving labels from parent folders.", flush=True)
    raw_df = pd.DataFrame([
        {"image_path": str(p), "label": p.parent.name}
        for p in image_files
    ])

raw_df = raw_df.drop_duplicates(subset=["image_path"]).reset_index(drop=True)
if raw_df.empty:
    raise RuntimeError(
        "No images were found. Check that the Kaggle dataset downloaded correctly "
        "and that the dataset files are present under dataset_root."
    )

print(f"Total valid RGB plant images: {len(raw_df):,}", flush=True)
print(f"Distinct species classes: {raw_df['label'].nunique():,}", flush=True)
print(f"Sample image: {raw_df.iloc[0]['image_path']}", flush=True)
print("Dataset indexing complete.", flush=True)


Starting LeafSnap dataset download/lookup...
If this is the first run, KaggleHub must download and unpack the dataset.
Using Colab cache for faster access to the 'leafsnap-dataset' dataset.
Dataset available at: /kaggle/input/leafsnap-dataset
Download/cache lookup took 0.1 minutes
Indexing image files (one pass)...
Reading metadata: /kaggle/input/leafsnap-dataset/leafsnap-dataset-images.txt
Total valid RGB plant images: 30,866
Distinct species classes: 185
Sample image: /kaggle/input/leafsnap-dataset/leafsnap-dataset/dataset/images/lab/abies_concolor/ny1157-01-1.jpg
Dataset indexing complete.


In [ ]:
# ============================================================
# 2. ONNX RUNTIME GPU & REMBG CONFIGURATION FOR TESLA T4
# Fixes: CUDNN_FE failure 8: HEURISTIC_QUERY_FAILED
# ============================================================

import onnxruntime as ort
from rembg import new_session, remove

print("=" * 70)
print("CONFIGURING ONNX RUNTIME CUDA EXECUTION PROVIDER")
print("=" * 70)
print(f"ONNX Runtime version: {ort.__version__}")
available_providers = ort.get_available_providers()
print(f"Available providers: {available_providers}")

REMBG_SESSION = None

if "CUDAExecutionProvider" in available_providers:
    try:
        # CRITICAL FIX FOR T4 GPU / cuDNN 9:
        # Using cudnn_conv_algo_search='DEFAULT' bypasses the broken cuDNN Frontend
        # heuristic execution plan search that triggers Conv_136 HEURISTIC_QUERY_FAILED.
        cuda_provider_options = {
            "device_id": 0,
            "arena_extend_strategy": "kSameAsRequested",
            "gpu_mem_limit": 8 * 1024 * 1024 * 1024,  # Maximum ONNX CUDA allocation budget (not a reservation)
            "cudnn_conv_algo_search": "DEFAULT",       # Fixes Conv HEURISTIC_QUERY_FAILED
            "do_copy_in_default_stream": True,
        }

        REMBG_SESSION = new_session(
            "u2net",
            providers=[("CUDAExecutionProvider", cuda_provider_options), "CPUExecutionProvider"]
        )

        active = REMBG_SESSION.inner_session.get_providers()
        print(f"Active providers for rembg: {active}")
        if "CUDAExecutionProvider" in active:
            print("✓ SUCCESS: rembg U2-Net session is utilizing Tesla T4 CUDA hardware acceleration!")
        else:
            print("⚠ Notice: Active provider fell back to CPU")

    except Exception as e:
        print(f"CUDA session configuration error: {e}")
        print("Creating standard session fallback...")
        REMBG_SESSION = new_session("u2net")
else:
    print("⚠ CUDAExecutionProvider not detected; creating CPU session")
    REMBG_SESSION = new_session("u2net")

# Test 1 dummy pass to verify no cuDNN Conv error
print("Testing U2-Net forward pass...")
dummy_img = np.zeros((224, 224, 3), dtype=np.uint8)
dummy_out = remove(dummy_img, session=REMBG_SESSION)
print(f"✓ Test inference successful! Output shape: {dummy_out.shape}")


CONFIGURING ONNX RUNTIME CUDA EXECUTION PROVIDER
ONNX Runtime version: 1.31.1
Available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


  0%|                                               | 0.00/176M [00:00<?, ?B/s]

Active providers for rembg: ['CUDAExecutionProvider', 'CPUExecutionProvider']
✓ SUCCESS: rembg U2-Net session is utilizing Tesla T4 CUDA hardware acceleration!
Testing U2-Net forward pass...
✓ Test inference successful! Output shape: (224, 224, 4)


In [ ]:
# ============================================================
# BACKGROUND REMOVAL — 8 WORKERS, RESUMABLE, CUDA + CPU
# Run after dataset indexing (Cell 3) and ONNX setup (Cell 4).
# ============================================================

import os
import gc
import time
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from rembg import remove

if "raw_df" not in globals() or raw_df is None or raw_df.empty:
    raise RuntimeError("Run the dataset indexing cell first; raw_df is missing/empty.")
if "REMBG_SESSION" not in globals() or REMBG_SESSION is None:
    raise RuntimeError("Run the ONNX/rembg configuration cell first.")

# These are worker threads, not eight GPUs. They overlap CPU image loading/saving
# with inference on the single available GPU. Too many concurrent GPU calls can
# reduce throughput or cause CUDA out-of-memory, so monitor speed and VRAM.
GPU_WORKERS = 8
CHUNK_SIZE = 32
CHECKPOINT_EVERY = 250
cv2.setNumThreads(1)

BASE_DIR = Path("/content/leafsnap_project")
PROCESSED_DIR = BASE_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
processed_index_path = PROCESSED_DIR / "image_index.csv"

raw_df = raw_df.copy()
raw_df["image_path"] = raw_df["image_path"].astype(str)
raw_df["label"] = raw_df["label"].astype(str)
raw_df = raw_df.drop_duplicates(subset=["image_path"]).reset_index(drop=True)

# Load checkpoint and normalize its columns.
if processed_index_path.exists():
    processed_df = pd.read_csv(processed_index_path)
else:
    processed_df = pd.DataFrame(columns=["original_path", "processed_path", "label"])

for col in ["original_path", "processed_path", "label"]:
    if col not in processed_df.columns:
        processed_df[col] = pd.Series(dtype="object")

processed_df = processed_df.drop_duplicates(subset=["original_path"], keep="last")
done_paths = set(processed_df["original_path"].dropna().astype(str))

# Resume only images that aren't already indexed with an existing output.
pending = []
recovered = []
for row in raw_df.itertuples(index=False):
    src = str(row.image_path)
    label = str(row.label)
    out_path = PROCESSED_DIR / label / f"{Path(src).stem}.png"

    if src in done_paths and out_path.is_file():
        continue
    if out_path.is_file() and out_path.stat().st_size > 0:
        recovered.append({
            "original_path": src,
            "processed_path": str(out_path),
            "label": label
        })
    else:
        pending.append((src, label))

if recovered:
    processed_df = pd.concat(
        [processed_df, pd.DataFrame(recovered)], ignore_index=True
    ).drop_duplicates(subset=["original_path"], keep="last")
    processed_df.to_csv(processed_index_path, index=False)

total_to_process = len(pending)
print(f"Dataset images: {len(raw_df):,}", flush=True)
print(f"Already indexed/recovered: {len(processed_df):,}", flush=True)
print(f"Remaining: {total_to_process:,}", flush=True)
print(f"Workers: {GPU_WORKERS}; chunk size: {CHUNK_SIZE}", flush=True)
try:
    print("Active ONNX providers:", REMBG_SESSION.inner_session.get_providers(), flush=True)
except Exception as e:
    print("Could not inspect active ONNX providers:", repr(e), flush=True)

def segment_leaf(src, label):
    """U2-Net inference through REMBG_SESSION; CPU does I/O and mask cleanup."""
    try:
        img_bgr = cv2.imread(src, cv2.IMREAD_COLOR)
        if img_bgr is None:
            return {"_error": f"Could not read image: {src}"}

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        # ONNX Runtime uses CUDA when CUDAExecutionProvider is active.
        rgba = remove(img_rgb, session=REMBG_SESSION)

        if rgba.ndim == 3 and rgba.shape[-1] == 4:
            mask = rgba[:, :, 3]
        else:
            # Defensive fallback for unexpected output format.
            mask = np.max(rgba, axis=2).astype(np.uint8)

        _, mask = cv2.threshold(mask, 10, 255, cv2.THRESH_BINARY)
        kernel = np.ones((3, 3), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

        contours, _ = cv2.findContours(
            mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        if contours:
            clean_mask = np.zeros_like(mask)
            cv2.drawContours(
                clean_mask, [max(contours, key=cv2.contourArea)],
                -1, 255, cv2.FILLED
            )
            mask = clean_mask

        # Save the RGB leaf on a black background, matching the original pipeline.
        result = np.zeros_like(img_rgb)
        result[mask > 0] = img_rgb[mask > 0]

        out_dir = PROCESSED_DIR / label
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"{Path(src).stem}.png"

        ok = cv2.imwrite(
            str(out_path),
            cv2.cvtColor(result, cv2.COLOR_RGB2BGR),
            [cv2.IMWRITE_PNG_COMPRESSION, 3]
        )
        if not ok:
            return {"_error": f"Failed to save image: {out_path}"}

        return {
            "original_path": src,
            "processed_path": str(out_path),
            "label": label
        }
    except Exception as exc:
        return {"_error": f"{src}: {type(exc).__name__}: {exc}"}

if total_to_process == 0:
    print("All images are already processed.", flush=True)
else:
    start_time = time.time()
    completed = 0
    failures = 0
    successful_records = []

    # Keep only one bounded chunk in flight to avoid excessive CPU RAM use.
    with ThreadPoolExecutor(max_workers=GPU_WORKERS) as pool:
        for offset in range(0, total_to_process, CHUNK_SIZE):
            chunk = pending[offset:offset + CHUNK_SIZE]
            futures = [pool.submit(segment_leaf, src, label) for src, label in chunk]

            for future in as_completed(futures):
                try:
                    result = future.result()
                except Exception as exc:
                    result = {"_error": f"Worker exception: {type(exc).__name__}: {exc}"}

                completed += 1
                if result and "_error" not in result:
                    successful_records.append(result)
                else:
                    failures += 1
                    if failures <= 5:
                        print("Processing error:", result.get("_error", "unknown error")[:300],
                              flush=True)

            # Save every chunk; avoids losing a large amount of work if Colab disconnects.
            if successful_records:
                processed_df = pd.concat(
                    [processed_df, pd.DataFrame(successful_records)],
                    ignore_index=True
                ).drop_duplicates(subset=["original_path"], keep="last")
                processed_df.to_csv(processed_index_path, index=False)
                successful_records.clear()

            elapsed = max(time.time() - start_time, 1e-9)
            speed = completed / elapsed
            eta_min = (total_to_process - completed) / max(speed * 60, 1e-9)
            print(
                f"[{completed:,}/{total_to_process:,}] {speed:.2f} img/s | "
                f"ETA {eta_min:.1f} min | failures {failures} | "
                f"indexed {len(processed_df):,}",
                flush=True
            )

    print("Background-removal stage complete.", flush=True)

print(f"Processed index: {processed_index_path}", flush=True)
print(f"Indexed rows: {len(processed_df):,}", flush=True)
print("Check `!nvidia-smi` during processing to monitor GPU utilization.", flush=True)

# Keep REMBG_SESSION available if you need to rerun this cell.
# Do not delete it here; later cells may need a deliberate cleanup before training.


Dataset images: 30,866
Already indexed/recovered: 0
Remaining: 30,866
Workers: 8; chunk size: 32
Active ONNX providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
[32/30,866] 8.49 img/s | ETA 60.5 min | failures 0 | indexed 32
[64/30,866] 8.84 img/s | ETA 58.1 min | failures 0 | indexed 64
[96/30,866] 8.82 img/s | ETA 58.2 min | failures 0 | indexed 96
[128/30,866] 8.92 img/s | ETA 57.5 min | failures 0 | indexed 128
[160/30,866] 8.81 img/s | ETA 58.1 min | failures 0 | indexed 160
[192/30,866] 8.82 img/s | ETA 58.0 min | failures 0 | indexed 192
[224/30,866] 8.91 img/s | ETA 57.3 min | failures 0 | indexed 224
[256/30,866] 8.91 img/s | ETA 57.2 min | failures 0 | indexed 256
[288/30,866] 8.86 img/s | ETA 57.5 min | failures 0 | indexed 288
[320/30,866] 8.78 img/s | ETA 58.0 min | failures 0 | indexed 320
[352/30,866] 8.71 img/s | ETA 58.4 min | failures 0 | indexed 352
[384/30,866] 8.76 img/s | ETA 58.0 min | failures 0 | indexed 384
[416/30,866] 8.75 img/s | ETA 58.0 min | fai

## 2. Botanical Leaf Feature Extraction
Botanical leaf taxonomy relies on 4 domain-specific feature groups:
- **Shape Morphology (14 Features)**: Area, perimeter, bounding aspect ratio, convex hull solidity, extent, equivalent diameter, circularity, normalized area, eccentricity, orientation angle, major/minor ellipse axes.
- **Texture & Pattern (13 Features)**: Mean pixel intensity, variance/std, Shannon entropy, and 10-bin Uniform Local Binary Pattern (LBP).
- **Color Profiling (12 Features)**: RGB means, RGB standard deviations, HSV color means, and HSV standard deviations inside the segmented leaf mask.
- **Venation Structural Features (5 Features)**: CLAHE-enhanced Canny edge density, morphological skeleton length, normalized skeleton length, skeleton branch junction points, and normalized branch count.

**Total Explicit Botanical Features per Leaf = 44**.

In [ ]:
# ============================================================
# 4. RESUMPTION / VERIFICATION OF BACKGROUND REMOVAL
# ============================================================

PROCESSED_DIR = Path("/content/leafsnap_project/processed")
processed_index_path = PROCESSED_DIR / "image_index.csv"

if processed_index_path.exists():
    processed_df = pd.read_csv(processed_index_path)
    print(f"Total segmented images cataloged in index: {len(processed_df)}")
    print(f"Distinct species classes present:          {processed_df['label'].nunique()}")
else:
    raise FileNotFoundError("Please run the background removal cell above first!")


Total segmented images cataloged in index: 30866
Distinct species classes present:          185


## 3. Botanical Feature Extraction Functions (44 Features)

In [ ]:
# ============================================================
# 5. BOTANICAL FEATURE EXTRACTION FUNCTIONS (44 FEATURES)
# ============================================================

import cv2
import numpy as np
from skimage.feature import local_binary_pattern
from skimage.morphology import skeletonize

# --- 1. SHAPE FEATURES (14) ---
def extract_shape_features(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return [0.0] * 14

    c = max(contours, key=cv2.contourArea)
    area = float(cv2.contourArea(c))
    perimeter = float(cv2.arcLength(c, True))
    x, y, w, h = cv2.boundingRect(c)
    aspect_ratio = float(w) / float(h) if h > 0 else 0.0

    hull = cv2.convexHull(c)
    hull_area = float(cv2.contourArea(hull))
    solidity = (area / hull_area) if hull_area > 0 else 0.0
    extent = (area / float(w * h)) if (w * h) > 0 else 0.0
    equivalent_diameter = float(np.sqrt(4.0 * area / np.pi)) if area > 0 else 0.0
    circularity = float((4.0 * np.pi * area) / (perimeter ** 2)) if perimeter > 0 else 0.0

    mask_area = float(mask.shape[0] * mask.shape[1])
    normalized_area = (area / mask_area) if mask_area > 0 else 0.0

    eccentricity = 0.0
    orientation = 0.0
    major_axis = 0.0
    minor_axis = 0.0

    if len(c) >= 5:
        try:
            ellipse = cv2.fitEllipse(c)
            (_, _), (axis1, axis2), angle = ellipse
            major_axis = float(max(axis1, axis2))
            minor_axis = float(min(axis1, axis2))
            if major_axis > 0:
                ratio = min(max(minor_axis / major_axis, 0.0), 1.0)
                eccentricity = float(np.sqrt(max(0.0, 1.0 - ratio ** 2)))
            orientation = float(angle)
        except Exception:
            pass

    return [
        area, perimeter, float(w), float(h), aspect_ratio,
        solidity, extent, equivalent_diameter, circularity,
        normalized_area, eccentricity, orientation, major_axis, minor_axis
    ]

# --- 2. TEXTURE FEATURES (13) ---
def extract_texture_features(gray_img, mask):
    pixels = gray_img[mask > 0]
    if len(pixels) < 10:
        return [0.0] * 13

    mean_intensity = float(np.mean(pixels))
    std_intensity = float(np.std(pixels))

    hist = np.bincount(pixels.ravel(), minlength=256).astype(np.float32)
    hist /= (hist.sum() + 1e-8)
    entropy = float(-np.sum(hist * np.log2(hist + 1e-8)))

    # Uniform LBP
    lbp = local_binary_pattern(gray_img, P=8, R=1, method="uniform")
    leaf_lbp = lbp[mask > 0]
    lbp_hist, _ = np.histogram(leaf_lbp, bins=np.arange(0, 11), density=False)
    lbp_hist = lbp_hist.astype(np.float32)
    lbp_hist /= (lbp_hist.sum() + 1e-8)

    return [
        mean_intensity, std_intensity, entropy,
        float(lbp_hist[0]), float(lbp_hist[1]), float(lbp_hist[2]),
        float(lbp_hist[3]), float(lbp_hist[4]), float(lbp_hist[5]),
        float(lbp_hist[6]), float(lbp_hist[7]), float(lbp_hist[8]),
        float(lbp_hist[9])
    ]

# --- 3. COLOR FEATURES (12) ---
def extract_color_features(rgb_img, mask):
    rgb_pixels = rgb_img[mask > 0]
    if len(rgb_pixels) == 0:
        return [0.0] * 12

    hsv_img = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2HSV)
    hsv_pixels = hsv_img[mask > 0]

    rgb_mean = np.mean(rgb_pixels, axis=0)
    rgb_std = np.std(rgb_pixels, axis=0)
    hsv_mean = np.mean(hsv_pixels, axis=0)
    hsv_std = np.std(hsv_pixels, axis=0)

    return [
        float(rgb_mean[0]), float(rgb_mean[1]), float(rgb_mean[2]),
        float(rgb_std[0]), float(rgb_std[1]), float(rgb_std[2]),
        float(hsv_mean[0]), float(hsv_mean[1]), float(hsv_mean[2]),
        float(hsv_std[0]), float(hsv_std[1]), float(hsv_std[2])
    ]

# --- 4. VEIN / STRUCTURAL FEATURES (5) ---
def extract_vein_features(gray_img, mask):
    leaf_pixels = gray_img[mask > 0]
    if len(leaf_pixels) == 0:
        return [0.0] * 5

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray_img)

    edges = cv2.Canny(enhanced, 50, 150)
    leaf_edges = cv2.bitwise_and(edges, edges, mask=mask)

    kernel = np.ones((3, 3), dtype=np.uint8)
    eroded_mask = cv2.erode(mask, kernel, iterations=1)
    inner_edges = cv2.bitwise_and(leaf_edges, leaf_edges, mask=eroded_mask)

    leaf_area = int(np.sum(eroded_mask > 0))
    if leaf_area == 0:
        return [0.0] * 5

    edge_density = float(np.sum(inner_edges > 0)) / leaf_area
    skeleton = skeletonize(inner_edges > 0)
    skeleton_length = float(np.sum(skeleton))
    norm_skeleton_length = skeleton_length / leaf_area

    skeleton_u8 = skeleton.astype(np.uint8)
    kernel_branch = np.array([[1, 1, 1], [1, 10, 1], [1, 1, 1]], dtype=np.uint8)
    neighbors = cv2.filter2D(skeleton_u8, -1, kernel_branch)
    branch_points = float(np.sum(neighbors >= 13))
    norm_branch_points = branch_points / leaf_area

    return [
        edge_density, skeleton_length, norm_skeleton_length,
        branch_points, norm_branch_points
    ]

# --- 5. COMBINED BOTANICAL EXTRACTOR (EXACTLY 44) ---
def extract_all_botanical_features(image_path, mask_path=None):
    img = cv2.imread(str(image_path))
    if img is None:
        return None

    rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    if mask_path is not None:
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            return None
    else:
        mask = (np.any(rgb_img > 0, axis=-1)).astype(np.uint8) * 255

    mask = np.where(mask > 0, 255, 0).astype(np.uint8)

    feats = (
        extract_shape_features(mask) +
        extract_texture_features(gray_img, mask) +
        extract_color_features(rgb_img, mask) +
        extract_vein_features(gray_img, mask)
    )

    if len(feats) != 44:
        raise ValueError(f"Feature count error: expected 44, got {len(feats)}")

    return [float(x) if np.isfinite(x) else 0.0 for x in feats]

print("✓ Botanical feature extraction functions initialized (44 features per image)")


✓ Botanical feature extraction functions initialized (44 features per image)


## 4. Batch Botanical Feature Extraction Pipeline

In [ ]:
# ============================================================
# 6. BATCH BOTANICAL FEATURE EXTRACTION PIPELINE
# Resumable + Checkpointed + Multi-Threaded / Fast
# ============================================================

import time
from concurrent.futures import ThreadPoolExecutor, as_completed

FEATURES_DIR = Path("/content/leafsnap_project/features")
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_CACHE_PATH = FEATURES_DIR / "botanical_features.csv"

if not processed_index_path.exists():
    raise FileNotFoundError(f"Index not found: {processed_index_path}")

processed_df = pd.read_csv(processed_index_path).drop_duplicates(subset=["processed_path"]).reset_index(drop=True)

# Load existing cache if present
EXPECTED_COLUMNS = ["processed_path"] + [f"f_{i}" for i in range(44)]

if FEATURE_CACHE_PATH.exists():
    features_df = pd.read_csv(FEATURE_CACHE_PATH)
    if len(features_df.columns) != 45:
        print("Cache column mismatch, rebuilding cache...")
        features_df = pd.DataFrame(columns=EXPECTED_COLUMNS)
else:
    features_df = pd.DataFrame(columns=EXPECTED_COLUMNS)

cached_paths = set(features_df["processed_path"].astype(str))
to_extract = processed_df[~processed_df["processed_path"].isin(cached_paths)].reset_index(drop=True)

print("=" * 70)
print("BOTANICAL FEATURE EXTRACTION")
print("=" * 70)
print(f"Total processed images:   {len(processed_df)}")
print(f"Already cached features:  {len(features_df)}")
print(f"Images to extract:        {len(to_extract)}")

def extract_single(row):
    p = str(row.processed_path)
    try:
        feats = extract_all_botanical_features(p)
        if feats is not None and len(feats) == 44:
            return [p] + [float(x) for x in feats]
    except Exception:
        pass
    return None

SAVE_INTERVAL = 250
new_feature_records = []
total = len(to_extract)

if total > 0:
    start_time = time.time()
    # Parallel CPU execution (uses both Colab CPU cores)
    with ThreadPoolExecutor(max_workers=multiprocessing.cpu_count()) as executor:
        futures = {executor.submit(extract_single, row): i for i, row in enumerate(to_extract.itertuples(index=False))}

        for count, future in enumerate(as_completed(futures), start=1):
            res = future.result()
            if res is not None:
                new_feature_records.append(res)

            if count % SAVE_INTERVAL == 0 or count == total:
                if new_feature_records:
                    temp_df = pd.DataFrame(new_feature_records, columns=EXPECTED_COLUMNS)
                    features_df = pd.concat([features_df, temp_df], ignore_index=True)
                    features_df = features_df.drop_duplicates(subset=["processed_path"], keep="last")
                    features_df.to_csv(FEATURE_CACHE_PATH, index=False)
                    new_feature_records = []

                elapsed = time.time() - start_time
                speed = count / elapsed if elapsed > 0 else 0
                eta_min = (total - count) / (speed * 60) if speed > 0 else 0
                print(f"[{count}/{total}] {speed:.2f} img/s | ETA: {eta_min:.1f} min | Cached: {len(features_df)}")

print("\n✓ 44-Feature extraction completed successfully!")
print(f"Feature cache saved to: {FEATURE_CACHE_PATH}")


BOTANICAL FEATURE EXTRACTION
Total processed images:   30866
Already cached features:  0
Images to extract:        30866


/tmp/ipykernel_2696/4088396231.py:67: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  features_df = pd.concat([features_df, temp_df], ignore_index=True)


[250/30866] 7.80 img/s | ETA: 65.4 min | Cached: 250
[500/30866] 7.85 img/s | ETA: 64.5 min | Cached: 500
[750/30866] 7.64 img/s | ETA: 65.7 min | Cached: 750
[1000/30866] 7.59 img/s | ETA: 65.6 min | Cached: 1000
[1250/30866] 7.68 img/s | ETA: 64.2 min | Cached: 1250
[1500/30866] 7.68 img/s | ETA: 63.7 min | Cached: 1500
[1750/30866] 7.63 img/s | ETA: 63.6 min | Cached: 1750
[2000/30866] 7.56 img/s | ETA: 63.6 min | Cached: 2000
[2250/30866] 7.55 img/s | ETA: 63.2 min | Cached: 2250
[2500/30866] 7.42 img/s | ETA: 63.7 min | Cached: 2500
[2750/30866] 7.37 img/s | ETA: 63.5 min | Cached: 2750
[3000/30866] 7.38 img/s | ETA: 62.9 min | Cached: 3000
[3250/30866] 7.37 img/s | ETA: 62.4 min | Cached: 3250
[3500/30866] 7.37 img/s | ETA: 61.9 min | Cached: 3500
[3750/30866] 7.39 img/s | ETA: 61.1 min | Cached: 3750
[4000/30866] 7.36 img/s | ETA: 60.9 min | Cached: 4000
[4250/30866] 7.33 img/s | ETA: 60.5 min | Cached: 4250
[4500/30866] 7.32 img/s | ETA: 60.0 min | Cached: 4500
[4750/30866] 7.3

In [ ]:
# ============================================================
# 7. DATASET MERGE, STRATIFIED SPLIT & FEATURE SCALING
# ============================================================

import json
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

processed_df = pd.read_csv(processed_index_path)
features_df = pd.read_csv(FEATURE_CACHE_PATH)

processed_df["processed_path"] = processed_df["processed_path"].astype(str)
features_df["processed_path"] = features_df["processed_path"].astype(str)

# Merge processed images and features
dataset_full = pd.merge(processed_df, features_df, on="processed_path", how="inner")

feature_cols = [f"f_{i}" for i in range(44)]
dataset_full = dataset_full.dropna(subset=["label"] + feature_cols).reset_index(drop=True)

# Verify files actually exist on disk
dataset_full = dataset_full[
    dataset_full["processed_path"].apply(lambda p: os.path.exists(p))
].reset_index(drop=True)

print("=" * 70)
print("MERGED DATASET SUMMARY")
print("=" * 70)
print(f"Total valid samples:     {len(dataset_full)}")
print(f"Distinct plant species:  {dataset_full['label'].nunique()}")
print(f"Botanical features:      {len(feature_cols)}")

# Class mappings
class_names = sorted(dataset_full["label"].unique())
num_classes = len(class_names)
label_to_id = {name: i for i, name in enumerate(class_names)}
id_to_label = {i: name for name, i in label_to_id.items()}

# Split: 70% Train, 15% Validation, 15% Test
try:
    X_train_val, X_test = train_test_split(
        dataset_full, test_size=0.15, stratify=dataset_full["label"], random_state=RANDOM_SEED
    )
    X_train, X_val = train_test_split(
        X_train_val, test_size=0.15 / 0.85, stratify=X_train_val["label"], random_state=RANDOM_SEED
    )
    print("✓ Stratified train/val/test split successful")
except Exception:
    X_train_val, X_test = train_test_split(dataset_full, test_size=0.15, random_state=RANDOM_SEED)
    X_train, X_val = train_test_split(X_train_val, test_size=0.15 / 0.85, random_state=RANDOM_SEED)
    print("⚠ Singletons detected: random train/val/test split applied")

X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

# Feature scaling strictly fit on train set
scaler = StandardScaler()
X_train_feats = scaler.fit_transform(X_train[feature_cols]).astype(np.float32)
X_val_feats = scaler.transform(X_val[feature_cols]).astype(np.float32)
X_test_feats = scaler.transform(X_test[feature_cols]).astype(np.float32)

# Numerical safety sanitization
X_train_feats = np.nan_to_num(X_train_feats, nan=0.0, posinf=0.0, neginf=0.0)
X_val_feats = np.nan_to_num(X_val_feats, nan=0.0, posinf=0.0, neginf=0.0)
X_test_feats = np.nan_to_num(X_test_feats, nan=0.0, posinf=0.0, neginf=0.0)

# Save metadata artifacts
with open(FEATURES_DIR / "feature_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open(MODELS_DIR / "class_names.json", "w") as f:
    json.dump(class_names, f, indent=2)

with open(FEATURES_DIR / "feature_columns.json", "w") as f:
    json.dump(feature_cols, f, indent=2)

y_train_idx = X_train["label"].map(label_to_id).astype(np.int32).values
y_val_idx = X_val["label"].map(label_to_id).astype(np.int32).values
y_test_idx = X_test["label"].map(label_to_id).astype(np.int32).values

print(f"\nTrain set:       {len(X_train)} samples")
print(f"Validation set:  {len(X_val)} samples")
print(f"Test set:        {len(X_test)} samples")
print("✓ Standard scaler and class mapping artifacts saved")


MERGED DATASET SUMMARY
Total valid samples:     30866
Distinct plant species:  185
Botanical features:      44
✓ Stratified train/val/test split successful

Train set:       21606 samples
Validation set:  4630 samples
Test set:        4630 samples
✓ Standard scaler and class mapping artifacts saved


## 5. High-Throughput TF.Data Pipeline

In [ ]:
# ============================================================
# 8. TF.DATA PIPELINE (TESLA T4 OPTIMIZED)
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers

AUTOTUNE = tf.data.AUTOTUNE

def load_image(image_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method="bilinear")
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return tf.cast(img, tf.float32)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(height_factor=(-0.10, 0.10), width_factor=(-0.10, 0.10)),
    layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
    layers.RandomContrast(0.10)
], name="leaf_augmentation")

# Baseline Dataset: Image Only
def create_dataset_baseline(image_paths, labels, is_training=False):
    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    ds = ds.map(lambda p, l: (load_image(p), l), num_parallel_calls=AUTOTUNE)

    if is_training:
        ds = ds.shuffle(buffer_size=min(len(image_paths), 2048), seed=RANDOM_SEED)
        ds = ds.map(lambda img, l: (data_augmentation(img, training=True), l), num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

# Fusion Dataset: (Image, 44 Botanical Features) -> Label
def create_dataset_fusion(image_paths, features, labels, is_training=False):
    ds = tf.data.Dataset.from_tensor_slices((image_paths, features, labels))
    ds = ds.map(lambda p, f, l: ((load_image(p), tf.cast(f, tf.float32)), l), num_parallel_calls=AUTOTUNE)

    if is_training:
        ds = ds.shuffle(buffer_size=min(len(image_paths), 2048), seed=RANDOM_SEED)
        # Botanical features are not augmented; only the visual image is augmented
        ds = ds.map(lambda inputs, l: ((data_augmentation(inputs[0], training=True), inputs[1]), l), num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

# Build datasets
train_ds_base = create_dataset_baseline(X_train["processed_path"].values, y_train_idx, is_training=True)
val_ds_base   = create_dataset_baseline(X_val["processed_path"].values, y_val_idx, is_training=False)
test_ds_base  = create_dataset_baseline(X_test["processed_path"].values, y_test_idx, is_training=False)

train_ds_fusion = create_dataset_fusion(X_train["processed_path"].values, X_train_feats, y_train_idx, is_training=True)
val_ds_fusion   = create_dataset_fusion(X_val["processed_path"].values, X_val_feats, y_val_idx, is_training=False)
test_ds_fusion  = create_dataset_fusion(X_test["processed_path"].values, X_test_feats, y_test_idx, is_training=False)

print("✓ Datasets successfully constructed for Tesla T4 GPU (Batch size: 32)")


✓ Datasets successfully constructed for Tesla T4 GPU (Batch size: 32)


## 6. EfficientNetV2S Architecture & Botanical Fusion Model

In [ ]:
# ============================================================
# 9. MODEL ARCHITECTURES: BASELINE & BOTANICAL FUSION
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers, models

def create_efficientnetv2s():
    base_model = tf.keras.applications.EfficientNetV2S(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )
    base_model.trainable = False
    return base_model

# ------------------------------------------------------------
# 1. BASELINE MODEL: EFFICIENTNETV2S ONLY
# ------------------------------------------------------------
def build_baseline_model(num_classes):
    base_model = create_efficientnetv2s()
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image_input")

    # Note: Do not hardcode training=False so Stage 2 fine-tuning can train unfrozen layers!
    x = base_model(inputs)
    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = layers.BatchNormalization(name="image_batch_norm")(x)
    x = layers.Dropout(DROPOUT, name="image_dropout")(x)
    x = layers.Dense(256, activation="relu", name="image_dense")(x)
    x = layers.Dropout(DROPOUT, name="classifier_dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32", name="classification_output")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="EfficientNetV2S_Baseline")
    return model, base_model

# ------------------------------------------------------------
# 2. BOTANICAL FEATURE FUSION MODEL
# Visual Backbone (256-D) + Botanical Branch (64-D) -> Classifier
# ------------------------------------------------------------
def build_fusion_model(num_classes, num_features=44):
    base_model = create_efficientnetv2s()

    # Visual Image Branch
    img_inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image_input")
    x = base_model(img_inputs)
    x = layers.GlobalAveragePooling2D(name="image_global_pool")(x)
    x = layers.BatchNormalization(name="image_batch_norm")(x)
    x = layers.Dropout(DROPOUT, name="image_dropout")(x)
    img_embedding = layers.Dense(256, activation="relu", name="image_embedding")(x)

    # Explicit Botanical Feature Branch (44 Features)
    feat_inputs = layers.Input(shape=(num_features,), name="botanical_input")
    y = layers.Dense(128, activation="relu", name="botanical_dense_128")(feat_inputs)
    y = layers.BatchNormalization(name="botanical_batch_norm_1")(y)
    y = layers.Dropout(DROPOUT, name="botanical_dropout_1")(y)
    y = layers.Dense(64, activation="relu", name="botanical_dense_64")(y)
    botanical_embedding = layers.BatchNormalization(name="botanical_embedding")(y)

    # Multimodal Fusion
    combined = layers.Concatenate(name="feature_fusion")([img_embedding, botanical_embedding])
    z = layers.Dense(256, activation="relu", name="fusion_dense_256")(combined)
    z = layers.BatchNormalization(name="fusion_batch_norm")(z)
    z = layers.Dropout(DROPOUT, name="fusion_dropout")(z)
    z = layers.Dense(128, activation="relu", name="fusion_dense_128")(z)
    z = layers.Dropout(DROPOUT, name="fusion_classifier_dropout")(z)
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32", name="classification_output")(z)

    model = models.Model(inputs=[img_inputs, feat_inputs], outputs=outputs, name="EfficientNetV2S_BotanicalFusion")
    return model, base_model

print("✓ Model architectures defined with mixed-precision output support")


✓ Model architectures defined with mixed-precision output support


## 7. Two-Stage Model Training with Mixed Precision on Tesla T4

In [ ]:
# ============================================================
# 10. TWO-STAGE MODEL TRAINING (TESLA T4 ACCELERATED)
# Stage 1: Transfer Learning (Dense Heads)
# Stage 2: Fine-Tuning (Top Backbone Layers)
# ============================================================

from tensorflow.keras import callbacks, optimizers

tf.config.optimizer.set_jit(True)  # Enable XLA

def train_model_pipeline(model, base_cnn, train_dataset, val_dataset, model_name):
    print("\n" + "=" * 70)
    print(f"STARTING TRAINING: {model_name}")
    print("=" * 70)

    best_model_path = MODELS_DIR / f"{model_name}.keras"

    checkpoint_cb = callbacks.ModelCheckpoint(
        filepath=str(best_model_path),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    )
    early_stop_cb = callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=4,
        restore_best_weights=True,
        verbose=1
    )
    lr_scheduler = callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )

    # --- STAGE 1: TRANSFER LEARNING ---
    print("\n--- STAGE 1: TRANSFER LEARNING (FROZEN BACKBONE) ---")
    base_cnn.trainable = False
    model.compile(
        optimizer=optimizers.AdamW(learning_rate=INITIAL_LR, weight_decay=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    t0 = time.time()
    h1 = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=EPOCHS,
        callbacks=[checkpoint_cb, early_stop_cb, lr_scheduler],
        verbose=1
    )
    print(f"Stage 1 completed in {(time.time() - t0) / 60:.2f} min")

    # --- STAGE 2: FINE-TUNING ---
    print("\n--- STAGE 2: FINE-TUNING TOP 20 LAYERS ---")
    base_cnn.trainable = True
    for layer in base_cnn.layers:
        layer.trainable = False

    fine_tune_count = min(20, len(base_cnn.layers))
    for layer in base_cnn.layers[-fine_tune_count:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True

    model.compile(
        optimizer=optimizers.AdamW(learning_rate=FINE_TUNE_LR, weight_decay=1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    t1 = time.time()
    h2 = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=FINE_TUNE_EPOCHS,
        callbacks=[checkpoint_cb, early_stop_cb, lr_scheduler],
        verbose=1
    )
    print(f"Stage 2 completed in {(time.time() - t1) / 60:.2f} min")

    if best_model_path.exists():
        best_model = tf.keras.models.load_model(best_model_path)
    else:
        best_model = model

    best_val_acc = max(h1.history.get("val_accuracy", [0]) + h2.history.get("val_accuracy", [0]))
    print(f"\n✓ {model_name} best validation accuracy: {best_val_acc:.4f}")
    return best_model, h1, h2

# ------------------------------------------------------------
# 1. TRAIN BASELINE MODEL
# ------------------------------------------------------------
tf.keras.backend.clear_session()
gc.collect()

base_model, base_backbone = build_baseline_model(num_classes)
base_model, h_base_1, h_base_2 = train_model_pipeline(
    base_model, base_backbone, train_ds_base, val_ds_base, "efficientnetv2s_baseline"
)

del base_model
del base_backbone
tf.keras.backend.clear_session()
gc.collect()

# ------------------------------------------------------------
# 2. TRAIN BOTANICAL FUSION MODEL
# ------------------------------------------------------------
fusion_model, fusion_backbone = build_fusion_model(num_classes, num_features=44)
fusion_model, h_fus_1, h_fus_2 = train_model_pipeline(
    fusion_model, fusion_backbone, train_ds_fusion, val_ds_fusion, "efficientnetv2s_botanical_fusion"
)

print("\n✓ All models successfully trained and checkpoints saved to:", MODELS_DIR)


82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step

STARTING TRAINING: efficientnetv2s_baseline

--- STAGE 1: TRANSFER LEARNING (FROZEN BACKBONE) ---
Epoch 1/20
676/676 ━━━━━━━━━━━━━━━━━━━━ 0s 641ms/step - accuracy: 0.1760 - loss: 3.9507
Epoch 1: val_accuracy improved from None to 0.41188, saving model to /content/leafsnap_project/models/efficientnetv2s_baseline.keras

Epoch 1: finished saving model to /content/leafsnap_project/models/efficientnetv2s_baseline.keras
676/676 ━━━━━━━━━━━━━━━━━━━━ 537s 711ms/step - accuracy: 0.2632 - loss: 3.1718 - val_accuracy: 0.4119 - val_loss: 2.2575 - learning_rate: 0.0010
Epoch 2/20
676/676 ━━━━━━━━━━━━━━━━━━━━ 0s 596ms/step - accuracy: 0.3953 - loss: 2.2706
Epoch 2: val_accuracy improved from 0.41188 to 0.45724, saving model to /content/leafsnap_project/models/efficientnetv2s_baseline.keras

Epoch 2: finished saving model to /content/leafsnap_project/models/efficientnetv2s_baseline.keras
676/676 ━━━━━━━━━━━━━━━━━━━━ 471s 622ms/step - accuracy: 0.418

## 8. Final Model Evaluation & Comparison

In [ ]:
# ============================================================
# 11. MODEL EVALUATION & QUANTITATIVE COMPARISON
# ============================================================

import seaborn as sns
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

def evaluate_performance(model, test_ds, y_true, class_names, model_name):
    print("\n" + "=" * 70)
    print(f"EVALUATING: {model_name}")
    print("=" * 70)

    predictions = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(predictions, axis=1)

    acc = accuracy_score(y_true, y_pred)
    macro_p = precision_score(y_true, y_pred, average="macro", zero_division=0)
    weighted_p = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    macro_r = recall_score(y_true, y_pred, average="macro", zero_division=0)
    weighted_r = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    top5_preds = np.argsort(predictions, axis=1)[:, -5:]
    top5_acc = float(np.mean([t in p for t, p in zip(y_true, top5_preds)]))

    metrics = {
        "Model": model_name,
        "Accuracy": acc,
        "Macro Precision": macro_p,
        "Weighted Precision": weighted_p,
        "Macro Recall": macro_r,
        "Weighted Recall": weighted_r,
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1,
        "Top-5 Accuracy": top5_acc
    }

    print(f"Accuracy:           {acc:.4f}")
    print(f"Macro F1:           {macro_f1:.4f}")
    print(f"Weighted F1:        {weighted_f1:.4f}")
    print(f"Top-5 Accuracy:     {top5_acc:.4f}")

    # Save classification report
    rep = classification_report(y_true, y_pred, labels=np.arange(len(class_names)), target_names=class_names, zero_division=0)
    with open(RESULTS_DIR / f"{model_name}_classification_report.txt", "w") as f:
        f.write(rep)

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(RESULTS_DIR / f"{model_name}_confusion_matrix.csv")

    # Plot
    plt.figure(figsize=(16, 13))
    sns.heatmap(cm, cmap="Blues", cbar=True)
    plt.title(f"Confusion Matrix: {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.savefig(RESULTS_DIR / f"{model_name}_confusion_matrix.png", dpi=180, bbox_inches="tight")
    plt.close()

    return metrics

# Evaluate both models
comparison_results = []

base_path = MODELS_DIR / "efficientnetv2s_baseline.keras"
if base_path.exists():
    m_base = tf.keras.models.load_model(base_path)
    comparison_results.append(evaluate_performance(m_base, test_ds_base, y_test_idx, class_names, "EfficientNetV2S_Baseline"))
    del m_base
    tf.keras.backend.clear_session()
    gc.collect()

fusion_path = MODELS_DIR / "efficientnetv2s_botanical_fusion.keras"
if fusion_path.exists():
    m_fus = tf.keras.models.load_model(fusion_path)
    comparison_results.append(evaluate_performance(m_fus, test_ds_fusion, y_test_idx, class_names, "EfficientNetV2S_Botanical_Fusion"))
    del m_fus
    tf.keras.backend.clear_session()
    gc.collect()

if comparison_results:
    comp_df = pd.DataFrame(comparison_results)
    comp_df.to_csv(RESULTS_DIR / "final_model_comparison.csv", index=False)
    print("\n" + "=" * 70)
    print("FINAL RESEARCH PERFORMANCE COMPARISON")
    print("=" * 70)
    display(comp_df.style.format({
        "Accuracy": "{:.4f}", "Macro Precision": "{:.4f}", "Weighted Precision": "{:.4f}",
        "Macro Recall": "{:.4f}", "Weighted Recall": "{:.4f}", "Macro F1": "{:.4f}",
        "Weighted F1": "{:.4f}", "Top-5 Accuracy": "{:.4f}"
    }))


## 9. Single-Leaf Plant Species Prediction with Visual Explanation

In [ ]:
# ============================================================
# 12. SINGLE-LEAF PLANT SPECIES PREDICTION
# ============================================================

fusion_model_path = MODELS_DIR / "efficientnetv2s_botanical_fusion.keras"
class_names_path = MODELS_DIR / "class_names.json"
scaler_path = FEATURES_DIR / "feature_scaler.pkl"

if not (fusion_model_path.exists() and class_names_path.exists() and scaler_path.exists()):
    raise FileNotFoundError("Required trained artifacts not found. Complete previous cells first!")

print("Loading trained botanical fusion model and scaler...")
fusion_model = tf.keras.models.load_model(fusion_model_path)
with open(class_names_path, "r") as f:
    class_names = np.asarray(json.load(f))
with open(scaler_path, "rb") as f:
    feature_scaler = pickle.load(f)

def predict_plant_species(image_path):
    orig = cv2.imread(str(image_path))
    if orig is None:
        print(f"Could not open image: {image_path}")
        return

    orig_rgb = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)

    # 1. Background removal using Tesla T4
    clean_png = remove(orig_rgb, session=REMBG_SESSION)
    mask = (clean_png[:, :, 3] > 10).astype(np.uint8) * 255 if clean_png.shape[-1] == 4 else (np.any(clean_png > 0, axis=-1)).astype(np.uint8) * 255

    clean_rgb = np.zeros_like(orig_rgb)
    clean_rgb[mask > 0] = orig_rgb[mask > 0]

    tmp_p = BASE_DIR / "tmp_eval.png"
    cv2.imwrite(str(tmp_p), cv2.cvtColor(clean_rgb, cv2.COLOR_RGB2BGR))

    # 2. Extract 44 explicit botanical features
    feats = extract_all_botanical_features(str(tmp_p))
    feats_scaled = feature_scaler.transform(np.asarray(feats, dtype=np.float32).reshape(1, -1))

    # 3. Model forward pass
    img_t = tf.expand_dims(load_image(str(tmp_p)), axis=0)
    probs = fusion_model.predict([img_t, feats_scaled], verbose=0)[0]

    top5_idx = np.argsort(probs)[-5:][::-1]
    best_species = class_names[top5_idx[0]]
    best_conf = probs[top5_idx[0]] * 100

    print("=" * 70)
    print(f"PREDICTION RESULT: {Path(image_path).name}")
    print("=" * 70)
    print(f"🌿 Identified Species: {best_species}")
    print(f"🎯 Confidence:         {best_conf:.2f}%")
    print("\nTop-5 Predictions:")
    for rank, idx in enumerate(top5_idx, start=1):
        print(f"  {rank}. {class_names[idx]:<35} {probs[idx]*100:.2f}%")

    # Visual explanation
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].imshow(orig_rgb)
    axes[0].set_title("Original Leaf Image")
    axes[0].axis("off")

    axes[1].imshow(clean_rgb)
    axes[1].set_title(f"Identified: {best_species}\n({best_conf:.1f}% confidence)")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

# Test on first available raw image
sample_test_img = raw_df.iloc[0]["image_path"]
predict_plant_species(sample_test_img)


## 10. Conclusion & Artifact Summary
The pipeline has successfully completed:
- Background removal with ONNX Runtime on Tesla T4 GPU
- 44-feature botanical morphological, textural, color, and venation extraction
- EfficientNetV2S + Botanical Feature Fusion training with Mixed Precision
- Checkpointed weights and evaluation matrices saved to `/content/leafsnap_project/`.